# Preprocessing Pipeline with Reusable src Modules

This notebook prepares the reusable preprocessing configuration for the ICU mortality project.

It now uses `src.features.preprocessing`, `src.data`, and `src.config` rather than defining the preprocessing pipeline inside the notebook.

**Outputs used by later notebooks**
- `models/preprocessor.joblib`
- `models/feature_config.json`
- validated `train_modeling_raw.csv`, `validation_modeling_raw.csv`, and `test_modeling_raw.csv`

Preprocessing is fitted **only on the training set** to prevent leakage.

## 1. Bootstrap project imports

In [1]:
from pathlib import Path
import sys

# Make imports work whether VS Code starts the notebook from project root
# or from the notebooks/ directory.
cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_DIR = cwd
elif (cwd.parent / "src").exists():
    PROJECT_DIR = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing src/. "
        "Open this notebook from the clinical-outcome-prediction project."
    )

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project root:", PROJECT_DIR)
print("Python:", sys.executable)

Project root: C:\Projects\clinical-outcome-prediction
Python: c:\Projects\clinical-outcome-prediction\.venv\Scripts\python.exe


## 2. Imports from src

In [2]:
import json
import joblib
import pandas as pd

from src.config import (
    MODELS_DIR,
    PREPROCESSOR_PATH,
    FEATURE_CONFIG_PATH,
    TARGET_COLUMN,
)
from src.data import (
    load_modeling_splits,
    validate_cohort,
    validate_patient_split,
)
from src.features import (
    infer_feature_types,
    build_preprocessor,
    fit_preprocessor,
    get_transformed_feature_names,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)

MODELS_DIR.mkdir(parents=True, exist_ok=True)

## 3. Load and validate patient-level splits

In [3]:
train_df, validation_df, test_df = load_modeling_splits()

for dataframe, name in [
    (train_df, "train"),
    (validation_df, "validation"),
    (test_df, "test"),
]:
    validate_cohort(
        dataframe,
        target_column=TARGET_COLUMN,
        dataframe_name=name,
    )

validate_patient_split(
    train_df,
    validation_df,
    test_df,
)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)
print("Patient-level split validation passed.")

Train: (89, 73)
Validation: (20, 73)
Test: (19, 73)
Patient-level split validation passed.


## 4. Define modeling features

In [4]:
# These columns identify patients/stays or occur outside the predictor set.
# They should not be passed to the model as ordinary predictors.
excluded_columns = {
    "subject_id",
    "hadm_id",
    "stay_id",
    TARGET_COLUMN,
    "intime",
    "outtime",
    "prediction_time",
}

# Additional leakage-prone columns are excluded if they happen to exist.
possible_leakage_columns = {
    "deathtime",
    "dischtime",
    "discharge_location",
    "hospital_los",
    "icu_los",
    "los",
}

excluded_columns |= possible_leakage_columns.intersection(train_df.columns)

retained_feature_columns = [
    column
    for column in train_df.columns
    if column not in excluded_columns
]

if not retained_feature_columns:
    raise ValueError("No modeling features were retained.")

numeric_features, categorical_features = infer_feature_types(
    train_df,
    retained_feature_columns,
)

print("Retained predictors:", len(retained_feature_columns))
print("Numeric predictors:", len(numeric_features))
print("Categorical predictors:", len(categorical_features))

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Retained predictors: 66
Numeric predictors: 59
Categorical predictors: 7

Numeric features:
['anchor_age', 'icu_admission_hour', 'weekend_admission', 'emergency_admission', 'transfer_admission', 'heart_rate_min', 'heart_rate_max', 'heart_rate_mean', 'respiratory_rate_min', 'respiratory_rate_max', 'respiratory_rate_mean', 'spo2_min', 'spo2_max', 'spo2_mean', 'sbp_min', 'sbp_max', 'sbp_mean', 'map_min', 'map_max', 'map_mean', 'temperature_c_min', 'temperature_c_max', 'temperature_c_mean', 'creatinine_first', 'creatinine_min', 'creatinine_max', 'bun_first', 'bun_min', 'bun_max', 'sodium_first', 'sodium_min', 'sodium_max', 'potassium_first', 'potassium_min', 'potassium_max', 'glucose_first', 'glucose_min', 'glucose_max', 'wbc_first', 'wbc_min', 'wbc_max', 'platelets_first', 'platelets_min', 'platelets_max', 'lactate_first', 'lactate_min', 'lactate_max', 'bilirubin_total_first', 'bilirubin_total_min', 'bilirubin_total_max', 'albumin_first', 'albumin_min', 'albumin_max', 'bicarbonate_first',

## 5. Check feature consistency across splits

In [5]:
for dataframe, split_name in [
    (validation_df, "validation"),
    (test_df, "test"),
]:
    missing = [
        column for column in retained_feature_columns
        if column not in dataframe.columns
    ]
    if missing:
        raise ValueError(
            f"{split_name} is missing modeling columns: {missing}"
        )

print("All modeling features are available in all three splits.")

All modeling features are available in all three splits.


## 6. Build and fit preprocessing pipeline on training data only

In [6]:
X_train = train_df[retained_feature_columns].copy()

preprocessor = build_preprocessor(
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    add_numeric_missing_indicators=True,
    scale_numeric_features=True,
    sparse_output=False,
)

fit_preprocessor(
    preprocessor,
    X_train,
)

feature_names_out = get_transformed_feature_names(
    preprocessor
)

print("Preprocessor fitted on training data only.")
print("Input feature count:", len(retained_feature_columns))
print("Transformed feature count:", len(feature_names_out))

Preprocessor fitted on training data only.
Input feature count: 66
Transformed feature count: 141


## 7. Transform each split as a validation check

In [7]:
for dataframe, split_name in [
    (train_df, "train"),
    (validation_df, "validation"),
    (test_df, "test"),
]:
    transformed = preprocessor.transform(
        dataframe[retained_feature_columns]
    )
    print(
        f"{split_name}: raw={dataframe.shape}, "
        f"transformed={transformed.shape}"
    )

print("Preprocessing validation passed.")

train: raw=(89, 73), transformed=(89, 141)
validation: raw=(20, 73), transformed=(20, 141)
test: raw=(19, 73), transformed=(19, 141)
Preprocessing validation passed.


## 8. Save preprocessor and feature configuration

In [8]:
feature_config = {
    "target_column": TARGET_COLUMN,
    "retained_feature_columns": retained_feature_columns,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "transformed_feature_count": int(len(feature_names_out)),
    "leakage_excluded_columns": sorted(excluded_columns),
    "preprocessing": {
        "numeric_imputation": "median",
        "numeric_missing_indicators": True,
        "numeric_scaling": "StandardScaler",
        "categorical_imputation": "most_frequent",
        "categorical_encoding": "OneHotEncoder(handle_unknown='ignore')",
        "fit_dataset": "training only",
    },
}

joblib.dump(
    preprocessor,
    PREPROCESSOR_PATH,
)

with open(
    FEATURE_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        feature_config,
        file,
        indent=2,
    )

print("Saved:", PREPROCESSOR_PATH)
print("Saved:", FEATURE_CONFIG_PATH)

Saved: C:\Projects\clinical-outcome-prediction\models\preprocessor.joblib
Saved: C:\Projects\clinical-outcome-prediction\models\feature_config.json


## 9. Reload and verify saved objects

In [9]:
loaded_preprocessor = joblib.load(
    PREPROCESSOR_PATH
)

with open(
    FEATURE_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as file:
    loaded_config = json.load(file)

assert loaded_config["target_column"] == TARGET_COLUMN
assert loaded_config["retained_feature_columns"] == retained_feature_columns

reloaded_output = loaded_preprocessor.transform(
    X_train.head(5)
)

assert reloaded_output.shape[1] == len(feature_names_out)

print("Saved preprocessing artifacts validated successfully.")

Saved preprocessing artifacts validated successfully.


## Notebook 5 summary

The preprocessing pipeline is now reusable from `src.features.preprocessing`.

Next: **Notebook 6 — Dummy Baseline and Logistic Regression**.